In [1]:
import pandas as pd

zillow_df = pd.read_csv('zillow.csv')

display(zillow_df.head(1))

,zpid,state_code,state_name,address,street,city,zipcode,price,price_formatted,beds,...,latitude,longitude,status,home_type,days_on_zillow,zestimate,detail_url,has_open_house,open_house_date,is_featured
0,17264897,CA,California,"979 Kevin Ave, Redlands, CA 92373",979 Kevin Ave,Redlands,92373,447000,"$447,000",3.0,...,34.04052,-117.186195,House for sale,SINGLE_FAMILY,5,455500.0,https://www.zillow.com/homedetails/979-Kevin-A...,False,NaN,False


In [2]:
zillow_df.columns

Index(['zpid', 'state_code', 'state_name', 'address', 'street', 'city',
       'zipcode', 'price', 'price_formatted', 'beds', 'baths', 'area_sqft',
       'latitude', 'longitude', 'status', 'home_type', 'days_on_zillow',
       'zestimate', 'detail_url', 'has_open_house', 'open_house_date',
       'is_featured'],
      dtype='object')

#### 1. State mismatch and drop columns
##### a. Resolve mismatches between state/state_code & address entries.
Manually inspecting some of the incorrect data links showed the address column is correct while the original split columns possessed incorrect information

##### b. Drop 'price_formatted' (there is another price column without $) and 'open_house_date' (there is a 'has_open_house' bool and most 'open_house_date' rows are empty).  Additionally, open house date does not reflect changes made after data scrape.

To resolve a:
* Drop state_name, and state_code, street, city, and zipcode
* Manually split them cleanly

To resolve b:
* Drop price_formatted, open_house_date

In [3]:
# dropping address related cols since many have missing information
# dropping price formatted since '$' is not useful for analysis
# dropping open_house_date since most rows are NaN

drop_cols = ['state_code', 'state_name', 'street', 'city', 'zipcode', 'price_formatted', 'open_house_date']

zillow_df.drop(columns=drop_cols, inplace=True)

display(zillow_df.head(1))

,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,home_type,days_on_zillow,zestimate,detail_url,has_open_house,is_featured
0,17264897,"979 Kevin Ave, Redlands, CA 92373",447000,3.0,2.0,1300.0,34.04052,-117.186195,House for sale,SINGLE_FAMILY,5,455500.0,https://www.zillow.com/homedetails/979-Kevin-A...,False,False


**Correction of 'state_name' and 'state_code' columns**

In [4]:
# Map state codes to full names
state_map = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia'
}

In [5]:
# two part split since there are two different delimiters

# example address column: 979 Kevin Ave, Redlands, CA 92373 splits into street address, City, State & Zipcode
address_parts = zillow_df['address'].str.split(', ', expand =True)

zillow_df['street_add'] = address_parts[0]
zillow_df['city'] = address_parts[1]
zillow_df['state_zipcode'] = address_parts[2]

# splits state code & zipcode (since no comma delimiter) into two columns
state_zip_split = zillow_df['state_zipcode'].str.split(' ', expand =True)
zillow_df['state_code'] = state_zip_split[0]
zillow_df['zipcode'] = state_zip_split[1]

# recreates the state_name column from the extracted mapped state_code
zillow_df['state_name'] = zillow_df['state_code'].map(state_map)

display(zillow_df.head(1))

,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,home_type,...,zestimate,detail_url,has_open_house,is_featured,street_add,city,state_zipcode,state_code,zipcode,state_name
0,17264897,"979 Kevin Ave, Redlands, CA 92373",447000,3.0,2.0,1300.0,34.04052,-117.186195,House for sale,SINGLE_FAMILY,...,455500.0,https://www.zillow.com/homedetails/979-Kevin-A...,False,False,979 Kevin Ave,Redlands,CA 92373,CA,92373,California


**At this point, the address parts have been corrected**

Example: There should not be any cols that say 500 Commonwealth Ave, Boston, MA 02135, but have state_name and state_code as Texas and TX

In [6]:
# save updated df for audit trail (if desired)
zillow_df.to_csv('zillow_corrected_address.csv')

#### 2. Removal of home_type rows that are not single family home variations

To resolve:
* Extract different home_type values and their frequencies
* Remove undesired columns

In [7]:
print(zillow_df['home_type'].value_counts() )

row_count_pre_home_type = zillow_df['home_type'].shape[0]

print()
print(f'There are {row_count_pre_home_type} rows in the home_type column')

home_type
SINGLE_FAMILY    2108
CONDO              30
TOWNHOUSE          25
MANUFACTURED       23
MULTI_FAMILY       19
LOT                 9
Name: count, dtype: int64

There are 2214 rows in the home_type column


In [8]:
zillow_df = (
    zillow_df[~zillow_df['home_type']
       .isin(['LOT', 'MANUFACTURED', 'MULTI_FAMILY']
        )
    ])

# confirms selected home_type rows are gone
print(zillow_df['home_type'].value_counts() )

row_count_post_home_type = zillow_df['home_type'].shape[0]

print()
print(f'Number of rows removed: {row_count_pre_home_type - row_count_post_home_type}')
print(f'There are {zillow_df.shape[0]} rows in the home_type column')

home_type
SINGLE_FAMILY    2108
CONDO              30
TOWNHOUSE          25
Name: count, dtype: int64

Number of rows removed: 51
There are 2163 rows in the home_type column


#### 3. Missing values cleaning

To resolve:
* Inspect Zillow dataset for missing value counts
* Drop zpid: 444753560 row (new development of homes)
* Drop rows with 'Undisclosed' street address

In [9]:
zillow_df.isnull().sum()

zpid                 0
address              0
price                0
beds                 0
baths                0
area_sqft           50
latitude             4
longitude            4
status               0
home_type            0
days_on_zillow       0
zestimate         1078
detail_url           0
has_open_house       0
is_featured          0
street_add           0
city                 0
state_zipcode        1
state_code           1
zipcode              1
state_name           1
dtype: int64

In [10]:
display(zillow_df[zillow_df['zipcode'].isna()])

,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,home_type,...,zestimate,detail_url,has_open_house,is_featured,street_add,city,state_zipcode,state_code,zipcode,state_name
2150,444753560,"Parker Plan, Bridle Creek",365900,3.0,2.0,1700.0,39.486683,-77.93531,New construction,SINGLE_FAMILY,...,NaN,https://www.zillow.com/community/bridle-creek/...,False,False,Parker Plan,Bridle Creek,None,None,None,NaN


* zpid: 444753560 is missing state_zipcode, state_code, zipcode, state_name.  It is a new development lot, not a single home, this row can be dropped 


In [11]:
# drop row since this is not an actual address
zillow_df.drop(
    zillow_df[
        zillow_df['zpid'] == 444753560].index, inplace=True
    )

Missing Values After Removal of zpid=444753560

* area_sqft           50
* latitude             4
* longitude            4
* zestimate         1077

Manual inspection of the dataset found multiple 'Undisclosed' addresses

* 455353390	(undisclosed Address), Seattle, WA 98112
* 460335453	Undisclosed, Canyon Creek, MT 59633
* 460335454	Undisclosed, Darby, MT 59829

In [12]:
# drops rows with undisclosed addresses (high value properties)
drop_list = [455353390, 460335453, 460335454]

zillow_df = zillow_df[~zillow_df['zpid'].isin(drop_list)]

print(f'Updated Null value count: \n{zillow_df.isnull().sum()[zillow_df.isnull().sum() > 0]}')
# zillow_df.isnull().sum()[zillow_df.isnull().sum() > 0]

Updated Null value count: 
area_sqft      50
latitude        1
longitude       1
zestimate    1074
dtype: int64


#### 4. Zipcode cleaning & zipcode/county matching

To resolve zipcode inconsistences:
* Check for zipcodes that are four numbers, not five.  These represent instances of leading 0's that are missing
* Impute leading 0's where necessary
* Remove instances of ZIP+4, keeping only the first five numbers of the zip code

To resolve zipcode/county matching:
* Use pgeocode's Nomination for zipcode lookups
* Create a function that queries Nomination and matches the zipcode and its corresponding county
* Saves the matching entry as a new column in the zillow dataset (necessary for county level property tax data joins)

In [13]:
# import property tax dataset
prop_tax_df = pd.read_csv('Property Taxes by State and County, 2026  Tax Foundation Maps.csv')

# check naming conventions
display(prop_tax_df.head(3))

,State,County,"Median Housing Value, 2024 ($)","Median Property Taxes Paid, 2024 ($)(5-Year Estimate)",Effective Property Tax Rate (2024)
0,Alabama,Autauga County,"$207,200",627,0.28%
1,Alabama,Baldwin County,"$316,900",960,0.29%
2,Alabama,Barbour County,"$112,900",462,0.31%


In [14]:
# converts zipcode column to a string and ensure all rows are are 5 digits, adding leading 0s where necessary
zillow_df['zipcode'] = zillow_df['zipcode'].astype(str).str.zfill(5)

# check for zipcodes with more than 5 digits
print(
    zillow_df['zipcode']
    .sort_values(key=lambda x: x.str.len(), ascending=False)
    .head(5)
)

1296    05839-9361
1311    05445-0000
1            91423
2            91606
3            92054
Name: zipcode, dtype: object


In [15]:
# removes the '-XXXX' from two rows
zillow_df['zipcode'] = (
    zillow_df['zipcode'].astype(str)
    .str[:5]
    .str.zfill(5)
)

# confirms index 1296 (05839-9361) and 1311 (05445-0000) successfully are updated
print(zillow_df.loc[1296, 'zipcode'])
print(zillow_df.loc[1311, 'zipcode'])

# confirms rows are successfully updated
print(
    zillow_df['zipcode']
    .sort_values(key=lambda x: x.str.len(), ascending=False)
    .head(5)
)

05839
05445
0    92373
1    91423
2    91606
3    92054
4    90042
Name: zipcode, dtype: object


In [16]:
import pgeocode

# start
n = pgeocode.Nominatim('us')

# zfill ensures 5 numbers, replciating from above to ensure no issues occur
# '+ County' bc property tax dataset uses 'XXXXX County'
def zip_to_county(zipcode):
    result = n.query_postal_code(str(zipcode).zfill(5))
    if result is not None and pd.notna(result['county_name']):
        return result['county_name'] + ' County'
    return None

zillow_df['county_lookup'] = zillow_df['zipcode'].apply(zip_to_county)

In [17]:
# Confirm new 'county_lookup' column is successfully created
display(zillow_df.head(1))

# New 'county_lookup' column is not showing any null values, all are populated
print(f'Null value count: \n{zillow_df.isnull().sum()[zillow_df.isnull().sum() > 0]}')


,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,home_type,...,detail_url,has_open_house,is_featured,street_add,city,state_zipcode,state_code,zipcode,state_name,county_lookup
0,17264897,"979 Kevin Ave, Redlands, CA 92373",447000,3.0,2.0,1300.0,34.04052,-117.186195,House for sale,SINGLE_FAMILY,...,https://www.zillow.com/homedetails/979-Kevin-A...,False,False,979 Kevin Ave,Redlands,CA 92373,CA,92373,California,San Bernardino County


Null value count: 
area_sqft      50
latitude        1
longitude       1
zestimate    1074
dtype: int64


In [18]:
zillow_df.to_csv('zillow_cleaned.csv') # save the new zillow_df if desired

In [19]:
# import state income dataset
state_inc_df = pd.read_csv('2026 State Income Tax Rates and Brackets  Tax Foundation.csv', skiprows = 1)

# inspect state income dataset structure
display(state_inc_df.head(3))

print()

# inspect previously loaded property tax dataset again
display(prop_tax_df.head(3))


,State,Single Filer (Rates),Unnamed: 2,Single Filer (Brackets),Married Filing Jointly (Rates),Unnamed: 5,Married Filing Jointly (Brackets),Standard Deduction (Single),Standard Deduction (Couple),Personal Exemption (Single),Personal Exemption (Couple),Personal Exemption (Dependent)
0,"Alabama (a, b, c, uu)",2%,>,$0.00,2%,>,$0.00,"$3,000.00","$8,500.00","$1,500.00","$3,000.00","$1,000.00"
1,- Alabama,4%,>,$500.00,4%,>,"$1,000.00",NaN,NaN,NaN,NaN,NaN
2,- Alabama,5%,>,"$3,000.00",5%,>,"$6,000.00",NaN,NaN,NaN,NaN,NaN


,State,County,"Median Housing Value, 2024 ($)","Median Property Taxes Paid, 2024 ($)(5-Year Estimate)",Effective Property Tax Rate (2024)
0,Alabama,Autauga County,"$207,200",627,0.28%
1,Alabama,Baldwin County,"$316,900",960,0.29%
2,Alabama,Barbour County,"$112,900",462,0.31%


#### 5. State Income Tax Dataset Cleaning

* Drop unnecessary columns
* Strip symbols
* Rename columns

In [20]:
import re
import numpy as np

print(state_inc_df.columns)

# columns to drop 
col_drop = [
    'Unnamed: 2',                       
    'Unnamed: 5',                       
    'Standard Deduction (Single)',
    'Standard Deduction (Couple)',
    'Personal Exemption (Single)',
    'Personal Exemption (Couple)',
    'Personal Exemption (Dependent)'
]

# dropping '>' columns and deduction columns 
state_inc_df = state_inc_df.drop(columns = col_drop, errors = 'ignore')

Index(['State', 'Single Filer (Rates)', 'Unnamed: 2',
       'Single Filer (Brackets)', 'Married Filing Jointly (Rates)',
       'Unnamed: 5', 'Married Filing Jointly (Brackets)',
       'Standard Deduction (Single)', 'Standard Deduction (Couple)',
       'Personal Exemption (Single)', 'Personal Exemption (Couple)',
       'Personal Exemption (Dependent)'],
      dtype='object')


In [21]:
# checking df with dropped columns 
display(state_inc_df.head(5))

,State,Single Filer (Rates),Single Filer (Brackets),Married Filing Jointly (Rates),Married Filing Jointly (Brackets)
0,"Alabama (a, b, c, uu)",2%,$0.00,2%,$0.00
1,- Alabama,4%,$500.00,4%,"$1,000.00"
2,- Alabama,5%,"$3,000.00",5%,"$6,000.00"
3,Alaska,none,NaN,none,NaN
4,"Arizona (e, f, u, vv)",3%,$0.00,3%,$0.00


In [22]:
# dropping $, %, and commas 
state_inc_df = state_inc_df.replace(['\$' , '\%', '\,'], '', regex = True)

# need to fix state names 
state_inc_df['State'] = (
    state_inc_df['State']
    .str.replace(r'\(.*?\)', '', regex = True)
    .str.replace('-', '', regex = False)
    .str.strip()
)

# count missing values for each column 
print("Missing values per column:")
print(state_inc_df.isna().sum())


Missing values per column:
State                                 0
Single Filer (Rates)                  5
Single Filer (Brackets)              14
Married Filing Jointly (Rates)        5
Married Filing Jointly (Brackets)    13
dtype: int64


In [23]:
# drop empty rows, rows with None, and n.a 
state_inc_df = state_inc_df.replace(
    ['None', 'none', 'n.a', 'n.a.', ''], 
    np.nan
)
state_inc_df = state_inc_df.dropna(how = 'any')

# fix column names 
state_inc_df.rename(columns = {'State' : 'state', 'Single Filer (Rates)' : 'single_filer_rates', 
                               'Single Filer (Brackets)' : 'single_filer_brackets', 
                               'Married Filing Jointly (Rates)' : 'married_filing_jointly_rates',
                               'Married Filing Jointly (Brackets)' : 'married_filing_jointly_brackets'}, inplace = True)


# checking clean df 
display(state_inc_df.head(5))

,state,single_filer_rates,single_filer_brackets,married_filing_jointly_rates,married_filing_jointly_brackets
0,Alabama,2,0.00,2,0.00
1,Alabama,4,500.00,4,1000.00
2,Alabama,5,3000.00,5,6000.00
4,Arizona,3,0.00,3,0.00
5,Arkansas,2,0.00,2,0.00


#### 6. Property Tax Dataset Cleaning

* Strip symbols ($ | , | %)
* Update column naming conventions
* Drop duplicate columns
* Export cleaned dfs

In [24]:
# removal of $ and commas
prop_tax_df['med_housing_value'] = pd.to_numeric(prop_tax_df['Median Housing Value, 2024 ($)'].str.replace(r'[$,]', '', regex = True), errors = 'coerce')

# removal of % symbol 
prop_tax_df['prop_tax_rate'] = pd.to_numeric(prop_tax_df['Effective Property Tax Rate (2024)'].str.replace('%', '', regex = True), errors = 'coerce')

# note: original columns were not dropped yet to ensure correct values 

# printing dataset after cleaning 
# print(prop_tax_df.head())

# update column naming conventions 
prop_tax_df.rename(columns = {'Median Property Taxes Paid, 2024 ($)(5-Year Estimate)' : 'med_prop_tax_paid',
                               'State' : 'state', 'County' : 'county'}, inplace = True)

# drop duplicate columns
col_drop = [
    'Median Housing Value, 2024 ($)',
    'Effective Property Tax Rate (2024)'
]

# dropping > column and deduction columns 
prop_tax_df = prop_tax_df.drop(columns = col_drop, errors = 'ignore')


In [25]:
# printing dataset after dropping
display(prop_tax_df.head(5))

,state,county,med_prop_tax_paid,med_housing_value,prop_tax_rate
0,Alabama,Autauga County,627,207200.0,0.28
1,Alabama,Baldwin County,960,316900.0,0.29
2,Alabama,Barbour County,462,112900.0,0.31
3,Alabama,Bibb County,298,145700.0,0.20
4,Alabama,Blount County,523,175200.0,0.27


In [26]:
# Save cleaned state income tax & property tax datasets
state_inc_df.to_csv('cleaned_state_income_tax.csv', index = False)
prop_tax_df.to_csv('cleaned_property_tax.csv', index = False)
print("Saved: cleaned_state_income_tax.csv")
print("Saved: cleaned_property_tax.csv")

Saved: cleaned_state_income_tax.csv
Saved: cleaned_property_tax.csv


#### 7. Implementation of Median Household Income Data

* Drop stateFlagCode
* Initial column naming convention corrections
* dType conversions & aggregations
* Merger of dfs

In [27]:
# import median household income by state dataset
med_household_inc_df = pd.read_csv('median-household-income-by-state-2026.csv')

# check first few rows of the dataset 
display(med_household_inc_df.head(5))

# dropping state flag code column 
med_household_inc_df = med_household_inc_df.drop(columns = 'stateFlagCode', errors = 'ignore')

# printing first few rows of dataset to check 
display(med_household_inc_df.head(5))


,stateFlagCode,state,MedianHouseholdIncomeInOnePersonHousehold_2023,MedianHouseholdIncomeInTwoPersonHousehold_2023,MedianHouseholdIncomeInThreePersonHousehold_2023,MedianHouseholdIncomeInFourPersonHousehold_2023
0,NaN,District of Columbia,75814.0,147683.0,183198.0,215972.0
1,NaN,Maryland,52519.0,109285.0,125724.0,150702.0
2,NaN,California,50251.0,101959.0,113943.0,128813.0
3,NaN,Colorado,49447.0,104126.0,121291.0,140677.0
4,NaN,New Jersey,48523.0,104651.0,130685.0,156487.0


,state,MedianHouseholdIncomeInOnePersonHousehold_2023,MedianHouseholdIncomeInTwoPersonHousehold_2023,MedianHouseholdIncomeInThreePersonHousehold_2023,MedianHouseholdIncomeInFourPersonHousehold_2023
0,District of Columbia,75814.0,147683.0,183198.0,215972.0
1,Maryland,52519.0,109285.0,125724.0,150702.0
2,California,50251.0,101959.0,113943.0,128813.0
3,Colorado,49447.0,104126.0,121291.0,140677.0
4,New Jersey,48523.0,104651.0,130685.0,156487.0


In [28]:
# fix column names 
med_household_inc_df.rename(columns = {'MedianHouseholdIncomeInOnePersonHousehold_2023' : 'median_household_income_one_person', 
                                       'MedianHouseholdIncomeInTwoPersonHousehold_2023' : 'median_household_income_two_person',
                                       'MedianHouseholdIncomeInThreePersonHousehold_2023' : 'median_household_income_three_person',
                                       'MedianHouseholdIncomeInFourPersonHousehold_2023' : 'median_household_income_four_person'}, inplace = True)

# printing first few rows of dataset to check 
display(med_household_inc_df.head(5))

,state,median_household_income_one_person,median_household_income_two_person,median_household_income_three_person,median_household_income_four_person
0,District of Columbia,75814.0,147683.0,183198.0,215972.0
1,Maryland,52519.0,109285.0,125724.0,150702.0
2,California,50251.0,101959.0,113943.0,128813.0
3,Colorado,49447.0,104126.0,121291.0,140677.0
4,New Jersey,48523.0,104651.0,130685.0,156487.0


In [29]:
# converting tax paid column to number 
prop_tax_df['med_prop_tax_paid'] = pd.to_numeric(
    prop_tax_df['med_prop_tax_paid'].astype(str).str.replace(r'[$,]', '', regex = True), 
    errors = 'coerce'
)

# property taxes are averaged per state so they don't duplicate zillow cities
prop_tax_state_avg = prop_tax_df.groupby('state', as_index = False).agg({
    'med_housing_value': 'mean',
    'prop_tax_rate': 'mean',
    'med_prop_tax_paid': 'mean' 
})

# rename the average columns
prop_tax_state_avg.rename(columns = {
    'med_housing_value': 'state_avg_housing_value',
    'prop_tax_rate': 'state_avg_prop_tax_rate'
}, inplace = True)

# zillow + property taxes 
merged_df = pd.merge(zillow_df, prop_tax_state_avg, left_on = 'state_name', right_on = 'state', how = 'left')\

# keep highest tax bracket 
state_inc_unique = state_inc_df.sort_values('single_filer_rates').drop_duplicates(subset = ['state'], keep = 'last')

# result + income taxes 
merged_df = pd.merge(merged_df, state_inc_unique, on = 'state', how = 'left')

# clean up the redundant 'state' column from the merge
if 'state' in merged_df.columns:
    merged_df.drop(columns = ['state'], inplace = True)


In [30]:
# export final dataset
merged_df.to_csv('master_zillow_with_taxes.csv', index = False)
print(f"\nSaved master_zillow_with_taxes.csv")
print(f'Total Rows: {len(merged_df)}')


Saved master_zillow_with_taxes.csv
Total Rows: 2159


In [31]:
# reading tax bracket column as a number 
state_inc_df['single_filer_brackets'] = pd.to_numeric(
    state_inc_df['single_filer_brackets'].astype(str).str.replace(r'[$,]', '', regex=True), 
    errors='coerce'
)

# convert all to math numbers 
prop_tax_df['med_prop_tax_paid'] = pd.to_numeric(prop_tax_df['med_prop_tax_paid'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
prop_tax_df['med_housing_value'] = pd.to_numeric(prop_tax_df['med_housing_value'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
prop_tax_df['prop_tax_rate'] = pd.to_numeric(prop_tax_df['prop_tax_rate'].astype(str).str.replace(r'[%,]', '', regex=True), errors='coerce')

# group by state and find the median to get rid of outliers
prop_tax_state_avg = prop_tax_df.groupby('state', as_index=False).agg({
    'med_housing_value': 'median',
    'prop_tax_rate': 'median',
    'med_prop_tax_paid': 'median' 
})

# average median income for 1 to 4 person households
income_cols = [
    'median_household_income_one_person', 
    'median_household_income_two_person',
    'median_household_income_three_person', 
    'median_household_income_four_person'
]
med_household_inc_df['state_avg_median_income'] = med_household_inc_df[income_cols].mean(axis=1)

# finding realistic tax bracket 
# merge tax dataset with our new average income data
tax_with_income = pd.merge(state_inc_df, med_household_inc_df[['state', 'state_avg_median_income']], on='state', how='left')

# keep only brackets where the state average income is >= the bracket start
valid_brackets = tax_with_income[tax_with_income['state_avg_median_income'] >= tax_with_income['single_filer_brackets']]

# keep the highest valid bracket remaining
state_inc_realistic = valid_brackets.sort_values('single_filer_brackets').drop_duplicates(subset=['state'], keep='last')

# rename the columns so they make sense in the master dataset
prop_tax_state_avg.rename(columns={
    'med_housing_value': 'state_median_housing_value',
    'prop_tax_rate': 'state_median_prop_tax_rate'
}, inplace=True)

# merge
final_merged_df = pd.merge(zillow_df, prop_tax_state_avg, left_on='state_name', right_on='state', how='left')

# add new realistic income taxes
final_merged_df = pd.merge(final_merged_df, state_inc_realistic, on='state', how='left')

# clean duplicate columns 
if 'state' in final_merged_df.columns:
    final_merged_df.drop(columns=['state'], inplace=True)

# export new dataset
final_merged_df.to_csv('new_master_dataset.csv', index=False)

print(f"\nSaved new_master_dataset.csv")
print(f"Total Rows: {len(final_merged_df)}\n")

# final result check 
print(final_merged_df[['state_name', 'price', 'state_avg_median_income', 'single_filer_rates']].head())


Saved new_master_dataset.csv
Total Rows: 2159

   state_name    price  state_avg_median_income single_filer_rates
0  California   447000                  98741.5                  9
1  California  2795000                  98741.5                  9
2  California  1718000                  98741.5                  9
3  California   899999                  98741.5                  9
4  California   995000                  98741.5                  9


#### 8. Manual Corrections

1. Blank entires in state_avg_median_income, tax bracket, and tax rate columns
2. Alterations of incorrect data (discovered in manual inspection of outliers) Example: a large home sold for extremely low amount relative to size



In [32]:
df = pd.read_csv('new_master_dataset.csv')

In [33]:
print(f"\nQA: Number of NA in state_avg_median_income: {df['state_avg_median_income'].isnull().sum()}")


QA: Number of NA in state_avg_median_income: 381


In [34]:
# Impute state median income for states with no income tax

# cols to use for calculations
income_cols = [
    'median_household_income_one_person',
    'median_household_income_two_person',
    'median_household_income_three_person',
    'median_household_income_four_person'
]

# copied dataset with no income tax states
no_tax_states = (
    med_household_inc_df[med_household_inc_df['state'].isin(
        ['Texas', 'Tennessee', 'Florida', 'Nevada']
        )
    ].copy()
)

no_tax_states['state_avg_median_income'] = no_tax_states[income_cols].mean(axis=1)

for _, row in no_tax_states.iterrows():
    mask = df['state_name'] == row['state']
    df.loc[mask , 'state_avg_median_income'] = row['state_avg_median_income']

# Data integrity check, confirm states are populated
print(
    df[df['state_name'].isin(['Texas', 'Tennessee', 'Florida', 'Nevada'])]
    [['state_name', 'state_avg_median_income']]
    .drop_duplicates()
    .sort_values('state_name')
)

# final QA check
print(f"\nQA: Remaining NA in state_avg_median_income: {df['state_avg_median_income'].isnull().sum()}")

    state_name  state_avg_median_income
609    Florida                 79891.25
615     Nevada                 83007.25
617  Tennessee                 77656.75
122      Texas                 81742.50

QA: Remaining NA in state_avg_median_income: 0


In [35]:
tax_cols = ['single_filer_rates', 'single_filer_brackets', 'married_filing_jointly_rates',
            'married_filing_jointly_brackets']

print(df[tax_cols].isnull().sum())

# fill missing with 0
# these the states that had missing state tax info do not have a state income tax (Texas, Tennessee, Nevada, etc.)
df[tax_cols] = df[tax_cols].fillna(0)

single_filer_rates                 381
single_filer_brackets              381
married_filing_jointly_rates       381
married_filing_jointly_brackets    381
dtype: int64


In [36]:
print(df[tax_cols].isnull().sum())

single_filer_rates                 0
single_filer_brackets              0
married_filing_jointly_rates       0
married_filing_jointly_brackets    0
dtype: int64


In [37]:
# rows to remove
# 920	459883086	1861 Glasgo Road, Griswold, CT 06351 | Building is an old church, no bedrooms or bathrooms listed 
# 1297	235257090	2053 Pikes Falls Road, Jamaica, VT 05343 | Tiny home, SQFT (265) could skew results
# 1755	40981856	228 Tiger Creek Rd, Roan Mountain, TN 37687 | Auction starting price is listed ($10,000)
# 1781	41873062	3650 River Rd, Decatur, TN 37322 | Auction starting price listed ($1)

# 2097	22609053	1113 Woodland Dr, Charleston, WV 25302 | Gutted home, not sold for actual value

# 144	29054064	2717 Laurel Valley Ln, Arlington, TX 76006 | home is an auction, falsely labeled as 'House for sale'
# 2092	22388069	1598 Campbell Dr, Huntington, WV 25705 | Gutted home, not sold for actual value

In [38]:
print(f'Number of rows {df.shape[0]}')

zpids_to_remove = [459883086, 235257090, 40981856, 41873062, 22609053, 29054064, 22388069]

df = df[~df['zpid'].isin(zpids_to_remove)]

print(f'Updated number of rows {df.shape[0]}')

Number of rows 2159
Updated number of rows 2152


In [39]:
# 2031	1923497	    330 11th Ave N, Hopkins, MN 55343 | price $1 --> $1,065,000
# 497	21912588	2916 Fairfield Dr, Edmond, OK 73012 | SQFT 0 --> SQFT 1451
# 498	21743321	3305 Fireside Cir, Norman, OK 73072 | SQFT 0 --> SQFT 2100
# 2201	460336976	215 Buffalo Creek Rd, Williamson, WV 25661 | SQFT 0 --> SQFT 1631
# 2167	304616329	1202 3rd Run Rd, Glenville, WV 26351 | SQFT 0 --> SQFT 2420 (Zillow home description)
# 524	22096033	14806 E Latimer St, Tulsa, OK 74116 | bathrooms 0 --> bathrooms 2

# 145	52107549	10210 Stidham Rd, Conroe, TX 77302 | status 'Auction --> 'House for sale' & price $100k --> $2,597,000

# Source for 2201: https://www.mystatemls.com/property/215-buffalo-creek-rd-wiliamson-wv-25661/11642390/

In [40]:
update_values  = {
    1923497: {'price': 1065000},
    21912588: {'area_sqft': 1451},
    21743321: {'area_sqft': 2100},
    460336976: {'area_sqft': 1631},
    304616329: {'area_sqft': 2420},
    22096033: {'baths': 2},
    52107549: {'status': 'House for sale', 'price': 2597000}

}

for zpid, updates in update_values.items():
    for col, val in updates.items():
        df.loc[df['zpid'] == zpid, col] = val

# Ensures no rows were deleted by accident (number should be the same as previous block)
print(f'Number of rows {df.shape[0]}')


Number of rows 2152


In [41]:
# Dups removal
print(f'Number of rows {df.shape[0]}')

# finds dups
dup_mask = df.duplicated(subset='zpid', keep=False)

# for audit if needed later, unsure if dups were accidently added in somehow previously
df_dups = df[dup_mask].sort_values('zpid').copy()
df_dups.to_csv('dups_zpid.csv', index=False)

# drops dups
df = df.drop_duplicates(subset='zpid', keep='first').copy()

print(f'Number of rows {df.shape[0]}')

Number of rows 2152
Number of rows 2101


In [42]:
# save new dataset
df.to_csv('dataset_final.csv')